In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



I0000 00:00:1786547585.592083   57052 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786547586.107400   57052 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786547589.079470   57052 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# new lag features with baseling rolling features

In [2]:
df = pd.read_csv("../Dataset/df_for_EDA.csv")

In [3]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,Lag_24,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24
0,1998-12-24 01:00:00,27213.0,1,3,12,24,52,1998,0,28570.0,27669.0,26498.0,29475.375000,30894.589286,2736.646326
1,1998-12-24 02:00:00,25643.0,2,3,12,24,52,1998,0,27213.0,26162.0,25147.0,29453.750000,30897.541667,2765.861628
2,1998-12-24 03:00:00,24907.0,3,3,12,24,52,1998,0,25643.0,25483.0,24574.0,29429.750000,30899.523810,2804.050165
3,1998-12-24 04:00:00,24721.0,4,3,12,24,52,1998,0,24907.0,25045.0,24393.0,29416.250000,30901.476190,2826.766154
4,1998-12-24 05:00:00,25144.0,5,3,12,24,52,1998,0,24721.0,25030.0,24860.0,29421.000000,30903.166667,2819.160745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,45787.0,42112.0,40343.500000,41856.327381,2295.270146
145194,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,45209.0,40797.0,40282.750000,41873.910714,2177.148998
145195,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,43663.0,38819.0,40230.208333,41895.238095,2106.081917
145196,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,41581.0,36287.0,40171.166667,41918.315476,2086.336995


In [4]:
df["Datetime"] = pd.to_datetime(df["Datetime"])


df["Hour"] = df["Datetime"].dt.hour

df["Day"] = df["Datetime"].dt.day

df["DayOfWeek"] = df["Datetime"].dt.dayofweek

df["Week"] = df["Datetime"].dt.isocalendar().week.astype(int)

df["Month"] = df["Datetime"].dt.month

df["Year"] = df["Datetime"].dt.year

df["IsWeekend"] = ( df["DayOfWeek"] >= 5 ).astype(int)

# Lag 

df["Lag_1"] = df["PJME_MW"].shift(1)
df["Lag_2"] = df["PJME_MW"].shift(2)
df["Lag_3"] = df["PJME_MW"].shift(3)
df["Lag_6"] = df["PJME_MW"].shift(6)
df["Lag_12"] = df["PJME_MW"].shift(12)
df["Lag_24"] = df["PJME_MW"].shift(24)
df["Lag_48"] = df["PJME_MW"].shift(48)
df["Lag_72"] = df["PJME_MW"].shift(72)
df["Lag_168"] = df["PJME_MW"].shift(168)

# Rolling mean and std

df["RollingMean_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .mean()
)


df["RollingMean_168"] = (
    df["PJME_MW"]
    .rolling(168)
    .mean()
)


df["RollingStd_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .std()
)

In [5]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,Lag_2,Lag_3,Lag_6,Lag_12,Lag_48,Lag_72
0,1998-12-24 01:00:00,27213.0,1,3,12,24,52,1998,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1998-12-24 02:00:00,25643.0,2,3,12,24,52,1998,0,27213.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998-12-24 03:00:00,24907.0,3,3,12,24,52,1998,0,25643.0,...,NaN,NaN,NaN,NaN,27213.0,NaN,NaN,NaN,NaN,NaN
3,1998-12-24 04:00:00,24721.0,4,3,12,24,52,1998,0,24907.0,...,NaN,NaN,NaN,NaN,25643.0,27213.0,NaN,NaN,NaN,NaN
4,1998-12-24 05:00:00,25144.0,5,3,12,24,52,1998,0,24721.0,...,NaN,NaN,NaN,NaN,24907.0,25643.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,42112.0,40343.500000,41856.327381,2295.270146,44147.0,41213.0,38726.0,40154.0,43308.0,45896.0
145194,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,40797.0,40282.750000,41873.910714,2177.148998,44343.0,44147.0,38737.0,40309.0,42440.0,45377.0
145195,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,38819.0,40230.208333,41895.238095,2106.081917,44284.0,44343.0,39337.0,39884.0,40661.0,44092.0
145196,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,36287.0,40171.166667,41918.315476,2086.336995,43751.0,44284.0,41213.0,39544.0,38207.0,42257.0


In [6]:
df.isna().sum()

Datetime             0
PJME_MW              0
Hour                 0
DayOfWeek            0
Month                0
Day                  0
Week                 0
Year                 0
IsWeekend            0
Lag_1                1
Lag_24              24
Lag_168            168
RollingMean_24      23
RollingMean_168    167
RollingStd_24       23
Lag_2                2
Lag_3                3
Lag_6                6
Lag_12              12
Lag_48              48
Lag_72              72
dtype: int64

In [7]:
missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().mean() * 100
)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

missing_summary

,Missing Count,Missing Percentage
Datetime,0,0.000000
PJME_MW,0,0.000000
Hour,0,0.000000
DayOfWeek,0,0.000000
Month,0,0.000000
Day,0,0.000000
Week,0,0.000000
Year,0,0.000000
IsWeekend,0,0.000000
Lag_1,1,0.000689


In [8]:
df = df.dropna().reset_index(drop=True)

In [9]:
df.to_csv("../Dataset/final_df.csv")

In [10]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,Lag_2,Lag_3,Lag_6,Lag_12,Lag_48,Lag_72
0,1998-12-17 01:00:00,29971.0,1,3,12,17,51,1998,0,32323.0,...,27213.0,35721.375000,31485.982143,3667.762970,35430.0,38234.0,40810.0,35867.0,29752.0,26437.0
1,1998-12-17 02:00:00,29046.0,2,3,12,17,51,1998,0,29971.0,...,25643.0,35676.666667,31506.238095,3744.754089,32323.0,35430.0,40317.0,35318.0,28489.0,24978.0
2,1998-12-17 03:00:00,28653.0,3,3,12,17,51,1998,0,29046.0,...,24907.0,35625.958333,31528.535714,3833.978618,29971.0,32323.0,39738.0,34817.0,27886.0,24353.0
3,1998-12-17 04:00:00,28774.0,4,3,12,17,51,1998,0,28653.0,...,24721.0,35573.875000,31552.660714,3920.893358,29046.0,29971.0,38234.0,34906.0,27712.0,24029.0
4,1998-12-17 05:00:00,29453.0,5,3,12,17,51,1998,0,28774.0,...,25144.0,35520.541667,31578.309524,3997.559484,28653.0,29046.0,35430.0,36661.0,28217.0,24363.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145025,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,42112.0,40343.500000,41856.327381,2295.270146,44147.0,41213.0,38726.0,40154.0,43308.0,45896.0
145026,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,40797.0,40282.750000,41873.910714,2177.148998,44343.0,44147.0,38737.0,40309.0,42440.0,45377.0
145027,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,38819.0,40230.208333,41895.238095,2106.081917,44284.0,44343.0,39337.0,39884.0,40661.0,44092.0
145028,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,36287.0,40171.166667,41918.315476,2086.336995,43751.0,44284.0,41213.0,39544.0,38207.0,42257.0


In [11]:
df.isna().sum()

Datetime           0
PJME_MW            0
Hour               0
DayOfWeek          0
Month              0
Day                0
Week               0
Year               0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
Lag_2              0
Lag_3              0
Lag_6              0
Lag_12             0
Lag_48             0
Lag_72             0
dtype: int64

In [10]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_48",
    "Lag_72"
]

target = "PJME_MW"

X = df[features]
y = df[target]

In [12]:
train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]
X_val = X.iloc[train_size:train_size + val_size]
X_test = X.iloc[train_size + val_size:]

y_train = y.iloc[:train_size]
y_val = y.iloc[train_size:train_size + val_size]
y_test = y.iloc[train_size + val_size:]

In [13]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)

X_val_scaled = X_scaler.transform(X_val)

X_test_scaled = X_scaler.transform(X_test)

In [14]:
import joblib

joblib.dump(
    X_scaler,
    "../Models/X_scaler.pkl"
)

['../Models/X_scaler.pkl']

In [15]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = y_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
)

In [16]:
import joblib

joblib.dump(
    y_scaler,
    "../Models/y_scaler.pkl"
)

['../Models/y_scaler.pkl']

In [16]:
print(X_train.isna().sum())
print(X_val.isna().sum())
print(X_test.isna().sum())

PJME_MW            0
Hour               0
Day                0
Week               0
Month              0
DayOfWeek          0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
Lag_2              0
Lag_3              0
Lag_6              0
Lag_12             0
Lag_48             0
Lag_72             0
dtype: int64
PJME_MW            0
Hour               0
Day                0
Week               0
Month              0
DayOfWeek          0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
Lag_2              0
Lag_3              0
Lag_6              0
Lag_12             0
Lag_48             0
Lag_72             0
dtype: int64
PJME_MW            0
Hour               0
Day                0
Week               0
Month              0
DayOfWeek          0
IsWeekend          0
Lag_1              0
Lag_24  

In [17]:
print("X_train inf:", np.isinf(X_train).sum().sum())
print("X_val inf:", np.isinf(X_val).sum().sum())
print("X_test inf:", np.isinf(X_test).sum().sum())

X_train inf: 0
X_val inf: 0
X_test inf: 0


In [18]:
print("X_train_scaled NaN:",
      np.isnan(X_train_scaled).sum())

print("X_val_scaled NaN:",
      np.isnan(X_val_scaled).sum())

print("X_test_scaled NaN:",
      np.isnan(X_test_scaled).sum())

X_train_scaled NaN: 0
X_val_scaled NaN: 0
X_test_scaled NaN: 0


In [19]:
print("X_train_scaled inf:",
      np.isinf(X_train_scaled).sum())

print("X_val_scaled inf:",
      np.isinf(X_val_scaled).sum())

print("X_test_scaled inf:",
      np.isinf(X_test_scaled).sum())

X_train_scaled inf: 0
X_val_scaled inf: 0
X_test_scaled inf: 0


In [20]:
def create_sequences(X, y, sequence_length, forecast_horizon):

    X_sequences = []
    y_sequences = []

    for i in range(
        sequence_length,
        len(X) - forecast_horizon + 1
    ):

        X_sequences.append(
            X[i-sequence_length:i]
        )

        y_sequences.append(
            y[i:i+forecast_horizon]
        )

    return np.array(X_sequences), np.array(y_sequences)

In [21]:
sequence_length = 48
forecast_horizon = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [77]:
print("X_train_seq NaN:",
      np.isnan(X_train_seq).sum())

print("y_train_seq NaN:",
      np.isnan(y_train_seq).sum())

print("X_val_seq NaN:",
      np.isnan(X_val_seq).sum())

print("y_val_seq NaN:",
      np.isnan(y_val_seq).sum())

X_train_seq NaN: 0
y_train_seq NaN: 0
X_val_seq NaN: 0
y_val_seq NaN: 0


In [78]:
X_train_seq[[1]]

array([[[-5.15692045e-01, -1.37279530e+00,  1.44294962e-01,
          1.60901014e+00,  1.57837541e+00,  5.86346421e-04,
         -6.31875576e-01, -3.73941864e-01, -3.51688443e-01,
         -1.03819985e+00,  7.23487081e-01, -2.30702218e-01,
         -4.00556596e-01, -1.34839168e-02,  4.62690110e-01,
          1.21161940e+00,  4.45382318e-01, -6.01707547e-01,
         -1.13991387e+00],
        [-5.75923910e-01, -1.22832131e+00,  1.44294962e-01,
          1.60901014e+00,  1.57837541e+00,  5.86346421e-04,
         -6.31875576e-01, -5.15710300e-01, -3.89855400e-01,
         -1.15105950e+00,  7.12249474e-01, -2.25027655e-01,
         -3.50956704e-01, -3.73961331e-01, -1.35019938e-02,
          1.12288080e+00,  3.68597573e-01, -6.94141272e-01,
         -1.23572709e+00],
        [-5.57379239e-01, -1.08384733e+00,  1.44294962e-01,
          1.60901014e+00,  1.57837541e+00,  5.86346421e-04,
         -6.31875576e-01, -5.75942728e-01, -3.66250133e-01,
         -1.17958110e+00,  7.00707149e-01, -2.

In [79]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [80]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1006 - mae: 0.2293 - val_loss: 0.0899 - val_mae: 0.2291
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0556 - mae: 0.1745 - val_loss: 0.0813 - val_mae: 0.2117
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0465 - mae: 0.1588 - val_loss: 0.0795 - val_mae: 0.2083
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0413 - mae: 0.1495 - val_loss: 0.0711 - val_mae: 0.1953
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0371 - mae: 0.1420 - val_loss: 0.0792 - val_mae: 0.2051
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0334 - mae: 0.1351 - val_loss: 0.0807 - val_mae: 0.2092
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0308 - mae: 0.1300 - val_loss: 0.0757 - val_mae: 0.1987
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0282 - mae: 0.1248 - val_loss: 0.0780 - val_mae: 0.2058
Epoch 9/10
1586/1586 ━━━━━━━━━━━

In [81]:
bilstm_pred_48 = bilstm_model_48.predict(
    X_test_seq
)

678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [82]:
bilstm_pred_48

array([[-0.01916628, -0.11162368, -0.1029131 , ..., -0.6244201 ,
        -0.55649674, -0.5953048 ],
       [-0.11098327, -0.14601988,  0.01467767, ..., -0.5620045 ,
        -0.5277519 , -0.6109145 ],
       [-0.03446794,  0.08518342,  0.33336633, ..., -0.54374725,
        -0.5576612 , -0.624085  ],
       ...,
       [ 1.4773215 ,  1.328861  ,  1.1779613 , ...,  2.5358148 ,
         2.3340333 ,  2.2212677 ],
       [ 1.16952   ,  1.0873945 ,  1.0776851 , ...,  2.4111333 ,
         2.0913062 ,  1.8290689 ],
       [ 1.0199214 ,  0.9878604 ,  1.0500486 , ...,  2.1658134 ,
         1.7268264 ,  1.416622  ]], dtype=float32)

# Metric Calculation block

In [22]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [84]:
bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48.reshape(-1, 1)
).reshape(bilstm_pred_48.shape)

In [23]:
y_test_original = y_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [87]:
bilstm_pred_original_48

array([[32285.725, 31682.459, 31739.295, ..., 28336.572, 28779.758,
        28526.543],
       [31686.639, 31458.031, 32506.549, ..., 28743.82 , 28967.312,
        28424.693],
       [32185.885, 32966.582, 34585.92 , ..., 28862.945, 28772.16 ,
        28338.76 ],
       ...,
       [42049.984, 41081.312, 40096.727, ..., 48956.43 , 47639.848,
        46904.074],
       [40041.65 , 39505.797, 39442.445, ..., 48142.906, 46056.105,
        44345.062],
       [39065.547, 38856.355, 39262.12 , ..., 46542.25 , 43677.953,
        41653.934]], dtype=float32)

In [89]:
results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

In [90]:
results

{'MAE': 1983.9874887875158,
 'MSE': 6996736.607804939,
 'RMSE': np.float64(2645.1345160133046),
 'MAPE': 6.674865726195193,
 'R2': 0.8323657998774167,
 'Bias': np.float64(939.277266175341)}

# Adding Drop out

In [93]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

         tf.keras.layers.Dropout(0.2),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [94]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 0.1179 - mae: 0.2520 - val_loss: 0.0854 - val_mae: 0.2214
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0683 - mae: 0.1970 - val_loss: 0.0782 - val_mae: 0.2113
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0600 - mae: 0.1841 - val_loss: 0.0816 - val_mae: 0.2182
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0550 - mae: 0.1764 - val_loss: 0.0746 - val_mae: 0.2012
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0512 - mae: 0.1703 - val_loss: 0.0743 - val_mae: 0.2032
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0480 - mae: 0.1652 - val_loss: 0.0785 - val_mae: 0.2086
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0457 - mae: 0.1613 - val_loss: 0.0807 - val_mae: 0.2128
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0435 - mae: 0.1576 - val_loss: 0.0734 - val_mae: 0.2004
Epoch 9/10
1586/1586 ━━━━━━━━━━━

In [95]:
bilstm_pred_48_dp = bilstm_model_48.predict(
    X_test_seq
)

678/678 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


In [96]:
bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_dp.reshape(-1, 1)
).reshape(bilstm_pred_48_dp.shape)

In [97]:
results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

In [98]:
results

{'MAE': 1942.9533460671366,
 'MSE': 7083305.258758675,
 'RMSE': np.float64(2661.4479628124755),
 'MAPE': 6.408207644955419,
 'R2': 0.8302917091445837,
 'Bias': np.float64(790.6286425787255)}

In [99]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

         tf.keras.layers.Dropout(0.3),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [100]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 0.1289 - mae: 0.2668 - val_loss: 0.0999 - val_mae: 0.2423
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0757 - mae: 0.2083 - val_loss: 0.0868 - val_mae: 0.2226
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0668 - mae: 0.1949 - val_loss: 0.0810 - val_mae: 0.2146
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0617 - mae: 0.1873 - val_loss: 0.0805 - val_mae: 0.2117
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0580 - mae: 0.1818 - val_loss: 0.0736 - val_mae: 0.2004
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0549 - mae: 0.1770 - val_loss: 0.0788 - val_mae: 0.2075
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0524 - mae: 0.1731 - val_loss: 0.0851 - val_mae: 0.2191
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0502 - mae: 0.1697 - val_loss: 0.0792 - val_mae: 0.2072
Epoch 9/10
1586/1586 ━━━━━━━━━━━

In [101]:
bilstm_pred_48_dp = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_dp.reshape(-1, 1)
).reshape(bilstm_pred_48_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


{'MAE': 1946.8032687056484,
 'MSE': 6910368.552649346,
 'RMSE': np.float64(2628.757986701961),
 'MAPE': 6.444103102086086,
 'R2': 0.8344350845530754,
 'Bias': np.float64(596.8464515947338)}

In [102]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

         tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [103]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_dp = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_dp.reshape(-1, 1)
).reshape(bilstm_pred_48_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1575 - mae: 0.2970 - val_loss: 0.0953 - val_mae: 0.2368
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0938 - mae: 0.2339 - val_loss: 0.0852 - val_mae: 0.2202
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0837 - mae: 0.2203 - val_loss: 0.0843 - val_mae: 0.2188
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0788 - mae: 0.2136 - val_loss: 0.0794 - val_mae: 0.2115
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0748 - mae: 0.2080 - val_loss: 0.0786 - val_mae: 0.2104
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0717 - mae: 0.2038 - val_loss: 0.0754 - val_mae: 0.2044
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0693 - mae: 0.2002 - val_loss: 0.0864 - val_mae: 0.2229
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0667 - mae: 0.1964 - val_loss: 0.0761 - val_mae: 0.2082
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 1824.9897190871102,
 'MSE': 6269557.12628878,
 'RMSE': np.float64(2503.9083701862533),
 'MAPE': 5.980970541785555,
 'R2': 0.8497882294417274,
 'Bias': np.float64(360.0020746323614)}

# Adding Batch Normaliation

In [104]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [105]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_bn = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_bn.reshape(-1, 1)
).reshape(bilstm_pred_48_bn.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.2508 - mae: 0.3692 - val_loss: 0.1058 - val_mae: 0.2486
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1172 - mae: 0.2631 - val_loss: 0.1000 - val_mae: 0.2440
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1012 - mae: 0.2435 - val_loss: 0.0880 - val_mae: 0.2273
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0926 - mae: 0.2326 - val_loss: 0.0924 - val_mae: 0.2364
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0875 - mae: 0.2258 - val_loss: 0.0932 - val_mae: 0.2357
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0827 - mae: 0.2193 - val_loss: 0.0824 - val_mae: 0.2180
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0786 - mae: 0.2140 - val_loss: 0.0854 - val_mae: 0.2241
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0750 - mae: 0.2090 - val_loss: 0.0933 - val_mae: 0.2364
Epoch 9/10
1586/1586 ━━━━━━━━

{'MAE': 2190.5832167536705,
 'MSE': 8063596.924461169,
 'RMSE': np.float64(2839.6473239578836),
 'MAPE': 7.38142629382956,
 'R2': 0.8068049869084544,
 'Bias': np.float64(1121.0317592042652)}

# New Optimizers

In [107]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [108]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_op = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_op.reshape(-1, 1)
).reshape(bilstm_pred_48_op.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 1.0120 - mae: 0.7823 - val_loss: 0.7873 - val_mae: 0.7122
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.8471 - mae: 0.7169 - val_loss: 0.6788 - val_mae: 0.6583
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.7285 - mae: 0.6665 - val_loss: 0.5826 - val_mae: 0.6062
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.6332 - mae: 0.6232 - val_loss: 0.5003 - val_mae: 0.5595
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.5558 - mae: 0.5856 - val_loss: 0.4345 - val_mae: 0.5213
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.4912 - mae: 0.5519 - val_loss: 0.3826 - val_mae: 0.4900
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.4370 - mae: 0.5213 - val_loss: 0.3417 - val_mae: 0.4640
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.3937 - mae: 0.4953 - val_loss: 0.3095 - val_mae: 0.4421
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2922.8521461639284,
 'MSE': 13810281.711359901,
 'RMSE': np.float64(3716.218738362948),
 'MAPE': 9.818812926488706,
 'R2': 0.6691206689746094,
 'Bias': np.float64(353.546182138574)}

In [109]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_op = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_op.reshape(-1, 1)
).reshape(bilstm_pred_48_op.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1564 - mae: 0.2979 - val_loss: 0.0912 - val_mae: 0.2272
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0983 - mae: 0.2390 - val_loss: 0.0926 - val_mae: 0.2274
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0882 - mae: 0.2259 - val_loss: 0.0829 - val_mae: 0.2133
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0821 - mae: 0.2179 - val_loss: 0.0851 - val_mae: 0.2226
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0778 - mae: 0.2119 - val_loss: 0.0882 - val_mae: 0.2274
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0748 - mae: 0.2076 - val_loss: 0.0819 - val_mae: 0.2172
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0723 - mae: 0.2040 - val_loss: 0.0914 - val_mae: 0.2299
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0703 - mae: 0.2012 - val_loss: 0.0862 - val_mae: 0.2275
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 1771.425878130128,
 'MSE': 5619109.234146657,
 'RMSE': np.float64(2370.4660373324605),
 'MAPE': 5.793036501472333,
 'R2': 0.8653722535707811,
 'Bias': np.float64(72.40864041789405)}

In [110]:
bilstm_model_48.save("/kaggle/working/bilstm_model_48_RMS_86.keras")

# Changing Learning Rate

In [111]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_lr = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_lr.reshape(-1, 1)
).reshape(bilstm_pred_48_lr.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.1556 - mae: 0.2966 - val_loss: 0.1123 - val_mae: 0.2528
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1102 - mae: 0.2524 - val_loss: 0.1045 - val_mae: 0.2426
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0988 - mae: 0.2391 - val_loss: 0.1062 - val_mae: 0.2472
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0925 - mae: 0.2314 - val_loss: 0.0992 - val_mae: 0.2394
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0882 - mae: 0.2260 - val_loss: 0.1193 - val_mae: 0.2707
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0843 - mae: 0.2208 - val_loss: 0.0991 - val_mae: 0.2400
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0815 - mae: 0.2172 - val_loss: 0.1275 - val_mae: 0.2852
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0796 - mae: 0.2148 - val_loss: 0.0900 - val_mae: 0.2260
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2130.885912750419,
 'MSE': 8005726.946277911,
 'RMSE': np.float64(2829.4393342635763),
 'MAPE': 6.992067660274932,
 'R2': 0.8081914886517158,
 'Bias': np.float64(446.30317291764544)}

In [112]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.005),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_lr = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_lr.reshape(-1, 1)
).reshape(bilstm_pred_48_lr.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.1398 - mae: 0.2822 - val_loss: 0.1102 - val_mae: 0.2516
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0981 - mae: 0.2378 - val_loss: 0.0976 - val_mae: 0.2358
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0869 - mae: 0.2236 - val_loss: 0.1156 - val_mae: 0.2672
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0813 - mae: 0.2161 - val_loss: 0.0969 - val_mae: 0.2428
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0770 - mae: 0.2104 - val_loss: 0.0912 - val_mae: 0.2336
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0734 - mae: 0.2056 - val_loss: 0.0857 - val_mae: 0.2190
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0712 - mae: 0.2027 - val_loss: 0.0868 - val_mae: 0.2243
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0691 - mae: 0.1999 - val_loss: 0.0853 - val_mae: 0.2215
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2082.0503937970416,
 'MSE': 7367050.406823056,
 'RMSE': np.float64(2714.231089429022),
 'MAPE': 7.015326774344978,
 'R2': 0.8234934839718115,
 'Bias': np.float64(767.3909411308093)}

In [113]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.00025),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_lr = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_lr.reshape(-1, 1)
).reshape(bilstm_pred_48_lr.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.2465 - mae: 0.3720 - val_loss: 0.1347 - val_mae: 0.2813
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1279 - mae: 0.2756 - val_loss: 0.1082 - val_mae: 0.2497
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1099 - mae: 0.2545 - val_loss: 0.1091 - val_mae: 0.2570
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1004 - mae: 0.2427 - val_loss: 0.0955 - val_mae: 0.2329
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0944 - mae: 0.2350 - val_loss: 0.0906 - val_mae: 0.2276
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0899 - mae: 0.2293 - val_loss: 0.0874 - val_mae: 0.2258
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0868 - mae: 0.2251 - val_loss: 0.0873 - val_mae: 0.2225
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0839 - mae: 0.2211 - val_loss: 0.1000 - val_mae: 0.2468
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2227.3588036346764,
 'MSE': 8440569.798233192,
 'RMSE': np.float64(2905.2658739318836),
 'MAPE': 7.506272166462795,
 'R2': 0.7977731268135365,
 'Bias': np.float64(720.3134669698136)}

# Early stopping

In [115]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [116]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    callbacks = [early_stopping],
    epochs=20,
    batch_size=64
)

bilstm_pred_48_er = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_er.reshape(-1, 1)
).reshape(bilstm_pred_48_er.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.1538 - mae: 0.2958 - val_loss: 0.1017 - val_mae: 0.2413
Epoch 2/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0986 - mae: 0.2395 - val_loss: 0.0951 - val_mae: 0.2388
Epoch 3/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0885 - mae: 0.2266 - val_loss: 0.0994 - val_mae: 0.2450
Epoch 4/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0825 - mae: 0.2187 - val_loss: 0.0793 - val_mae: 0.2121
Epoch 5/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0787 - mae: 0.2132 - val_loss: 0.1117 - val_mae: 0.2604
Epoch 6/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0754 - mae: 0.2087 - val_loss: 0.0813 - val_mae: 0.2166
Epoch 7/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0727 - mae: 0.2048 - val_loss: 0.0782 - val_mae: 0.2115
Epoch 8/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0703 - mae: 0.2017 - val_loss: 0.0801 - val_mae: 0.2123
Epoch 9/20
1586/1586 ━━━━━━━━━━━

{'MAE': 1829.4224290039122,
 'MSE': 6099368.564080156,
 'RMSE': np.float64(2469.6899732719808),
 'MAPE': 5.966216393770079,
 'R2': 0.853865762310984,
 'Bias': np.float64(84.7079144752107)}

# New Batch Size

In [117]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=32
)

bilstm_pred_48_bs = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_bs.reshape(-1, 1)
).reshape(bilstm_pred_48_bs.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.1390 - mae: 0.2809 - val_loss: 0.1098 - val_mae: 0.2586
Epoch 2/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0927 - mae: 0.2317 - val_loss: 0.0932 - val_mae: 0.2364
Epoch 3/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0839 - mae: 0.2200 - val_loss: 0.0830 - val_mae: 0.2150
Epoch 4/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0779 - mae: 0.2121 - val_loss: 0.0727 - val_mae: 0.2022
Epoch 5/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0741 - mae: 0.2067 - val_loss: 0.0791 - val_mae: 0.2153
Epoch 6/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0711 - mae: 0.2024 - val_loss: 0.0781 - val_mae: 0.2127
Epoch 7/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0680 - mae: 0.1981 - val_loss: 0.0790 - val_mae: 0.2113
Epoch 8/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0659 - mae: 0.1950 - val_loss: 0.0836 - val_mae: 0.2224
Epoch 9/15
3171/3171 ━━━━━━━━━━━

{'MAE': 1918.1593867896968,
 'MSE': 6381231.50726167,
 'RMSE': np.float64(2526.1099554971215),
 'MAPE': 6.426494760007206,
 'R2': 0.8471126327202941,
 'Bias': np.float64(787.9090135906599)}

In [118]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=16
)

bilstm_pred_48_bs = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_bs.reshape(-1, 1)
).reshape(bilstm_pred_48_bs.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 51s 8ms/step - loss: 0.1317 - mae: 0.2739 - val_loss: 0.1030 - val_mae: 0.2456
Epoch 2/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0904 - mae: 0.2285 - val_loss: 0.0778 - val_mae: 0.2114
Epoch 3/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - loss: 0.0816 - mae: 0.2168 - val_loss: 0.0835 - val_mae: 0.2175
Epoch 4/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0761 - mae: 0.2092 - val_loss: 0.0822 - val_mae: 0.2132
Epoch 5/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0718 - mae: 0.2033 - val_loss: 0.0747 - val_mae: 0.2020
Epoch 6/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0686 - mae: 0.1987 - val_loss: 0.0880 - val_mae: 0.2204
Epoch 7/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0659 - mae: 0.1947 - val_loss: 0.0796 - val_mae: 0.2106
Epoch 8/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0639 - mae: 0.1915 - val_loss: 0.0822 - val_mae: 0.2157
Epoch 9/15
6341/6341 ━━━━━━━━━━━

{'MAE': 1885.0681512219612,
 'MSE': 6352819.467577709,
 'RMSE': np.float64(2520.480007375125),
 'MAPE': 6.267438034918585,
 'R2': 0.8477933542928282,
 'Bias': np.float64(536.4699647170785)}

# Adding Addtional Layers

In [121]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                64,
                return_sequences=True
            )
        ),
    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(32)
        ),
    
        tf.keras.layers.BatchNormalization(),
    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Dense(64, activation="relu"),
    
        tf.keras.layers.Dense(24)
    ])

    return model

In [122]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64
)

bilstm_pred_48_Al = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_Al.reshape(-1, 1)
).reshape(bilstm_pred_48_Al.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.2291 - mae: 0.3591 - val_loss: 0.1438 - val_mae: 0.2983
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1349 - mae: 0.2830 - val_loss: 0.1134 - val_mae: 0.2612
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1198 - mae: 0.2663 - val_loss: 0.0961 - val_mae: 0.2374
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1106 - mae: 0.2555 - val_loss: 0.0975 - val_mae: 0.2392
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1050 - mae: 0.2488 - val_loss: 0.1093 - val_mae: 0.2508
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0992 - mae: 0.2420 - val_loss: 0.0841 - val_mae: 0.2172
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0949 - mae: 0.2367 - val_loss: 0.1197 - val_mae: 0.2601
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0912 - mae: 0.2323 - val_loss: 0.0895 - val_mae: 0.2295
Epoch 9/15
1586/1586 ━━━

{'MAE': 1975.2939845314254,
 'MSE': 7034777.14803121,
 'RMSE': np.float64(2652.3154314732647),
 'MAPE': 6.457290522288912,
 'R2': 0.8314543899029511,
 'Bias': np.float64(215.72153770373126)}

In [28]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                128,
                return_sequences=True
            )
        ),
    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Dense(64, activation="relu"),
    
        tf.keras.layers.Dense(24)
    ])

    return model

In [36]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64
)

bilstm_pred_48_Al = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_Al.reshape(-1, 1)
).reshape(bilstm_pred_48_Al.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.2079 - mae: 0.3346 - val_loss: 0.1620 - val_mae: 0.3248
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.1054 - mae: 0.2487 - val_loss: 0.0931 - val_mae: 0.2321
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0926 - mae: 0.2329 - val_loss: 0.0912 - val_mae: 0.2320
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0846 - mae: 0.2229 - val_loss: 0.0871 - val_mae: 0.2237
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - loss: 0.0786 - mae: 0.2147 - val_loss: 0.0881 - val_mae: 0.2236
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0731 - mae: 0.2076 - val_loss: 0.0839 - val_mae: 0.2169
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0686 - mae: 0.2012 - val_loss: 0.1196 - val_mae: 0.2731
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0647 - mae: 0.1955 - val_loss: 0.0838 - val_mae: 0.2171
Epoch 9/15
1586/1586 ━━━

{'MAE': 1794.6315722437075,
 'MSE': 5976450.495023555,
 'RMSE': np.float64(2444.6779941381965),
 'MAPE': 5.893069101172644,
 'R2': 0.8568107455713129,
 'Bias': np.float64(15.376053424087653)}

In [37]:
bilstm_model_48.save("/kaggle/working/bilstm_model_48_RMS_85_15bias.keras")

In [23]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense
from tensorflow.keras.optimizers import Adam

In [41]:
def build_bilstm_hyper(hp):

    model = Sequential()

    model.add(
        Bidirectional(
            LSTM(
                units=hp.Choice(
                    "lstm_units",
                    values=[32, 64, 128]
                )
            ),
            input_shape=(
                X_train_seq.shape[1],
                X_train_seq.shape[2]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    # Next 24 hours
    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [43]:
tuner_bilstm = kt.RandomSearch(
    build_bilstm_hyper,
    objective="val_loss",
    max_trials=5,
    directory="hyperparameter_tuning",
    project_name="bilstm_48_to_24"
)


tuner_bilstm.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 5 Complete [00h 03m 27s]
val_loss: 0.08911152929067612

Best val_loss So Far: 0.07462777197360992
Total elapsed time: 00h 17m 31s


In [46]:
best_hp_bilstm = tuner_bilstm.get_best_hyperparameters(
    num_trials=1
)[0]

best_hp_bilstm

In [47]:
print("BEST Bi-LSTM HYPERPARAMETERS")

print(
    "LSTM Units:",
    best_hp_bilstm.get("lstm_units")
)

print(
    "Dense Units:",
    best_hp_bilstm.get("dense_units")
)

print(
    "Learning Rate:",
    best_hp_bilstm.get("learning_rate")
)

BEST Bi-LSTM HYPERPARAMETERS
LSTM Units: 64
Dense Units: 128
Learning Rate: 0.0005


In [48]:
best_bilstm_model = tuner_bilstm.get_best_models(
    num_models=1
)[0]

best_bilstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        43,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,616 (244.59 KB)

 Trainable params: 62,616 (244.59 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
history_best_bilstm = best_bilstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0387 - mae: 0.1443 - val_loss: 0.0796 - val_mae: 0.2064
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0359 - mae: 0.1393 - val_loss: 0.0785 - val_mae: 0.2049
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0338 - mae: 0.1354 - val_loss: 0.0757 - val_mae: 0.1995
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0318 - mae: 0.1317 - val_loss: 0.0816 - val_mae: 0.2105
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0300 - mae: 0.1283 - val_loss: 0.0834 - val_mae: 0.2098
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0283 - mae: 0.1248 - val_loss: 0.0849 - val_mae: 0.2125
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0270 - mae: 0.1222 - val_loss: 0.0836 - val_mae: 0.2089
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0256 - mae: 0.1191 - val_loss: 0.0819 - val_mae: 0.2062
Epoch 9/15
1586/1586 ━━━━━━━━━━━

In [50]:
bilstm_pred_48_hy = best_bilstm_model.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_hy.reshape(-1, 1)
).reshape(bilstm_pred_48_hy.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


{'MAE': 2068.2570119985744,
 'MSE': 7906878.712426445,
 'RMSE': np.float64(2811.917266284064),
 'MAPE': 6.818685039150524,
 'R2': 0.8105597848366451,
 'Bias': np.float64(584.4459662288199)}

# LSTM 168

# LAG Features

In [24]:
sequence_length = 168
forecast_horizon = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [25]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [28]:
lstm_model = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

I0000 00:00:1786445378.160444      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786445378.163563      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1074 - mae: 0.2361 - val_loss: 0.0913 - val_mae: 0.2298
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0586 - mae: 0.1795 - val_loss: 0.0921 - val_mae: 0.2295
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0510 - mae: 0.1663 - val_loss: 0.0790 - val_mae: 0.2081
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0462 - mae: 0.1581 - val_loss: 0.0885 - val_mae: 0.2223
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0423 - mae: 0.1512 - val_loss: 0.0795 - val_mae: 0.2071
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0393 - mae: 0.1459 - val_loss: 0.0826 - val_mae: 0.2126
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0366 - mae: 0.1411 - val_loss: 0.0852 - val_mae: 0.2145
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0342 - mae: 0.1367 - val_loss: 0.0851 - val_mae: 0.2123
Epoch 9/10
1584/1584 ━━━━━━━━━━

In [31]:
lstm_pred_168 = lstm_model.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168.reshape(-1, 1)
).reshape(lstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2093.406195731432,
 'MSE': 8107396.7998581575,
 'RMSE': np.float64(2847.34908289415),
 'MAPE': 6.8988881241233635,
 'R2': 0.8062684460954157,
 'Bias': np.float64(792.6300895236636)}

# Drop Out

In [32]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [33]:
lstm_model_dp = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_dp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_dp.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1351 - mae: 0.2721 - val_loss: 0.1027 - val_mae: 0.2453
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0793 - mae: 0.2139 - val_loss: 0.0927 - val_mae: 0.2326
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0715 - mae: 0.2022 - val_loss: 0.0809 - val_mae: 0.2143
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0667 - mae: 0.1953 - val_loss: 0.0927 - val_mae: 0.2316
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0626 - mae: 0.1891 - val_loss: 0.0843 - val_mae: 0.2190
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0597 - mae: 0.1846 - val_loss: 0.0852 - val_mae: 0.2196
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0572 - mae: 0.1811 - val_loss: 0.0917 - val_mae: 0.2268
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0550 - mae: 0.1776 - val_loss: 0.0930 - val_mae: 0.2283
Epoch 9/10
1584/1584 ━━━━━━━━━

In [34]:
lstm_pred_168_dp = lstm_model.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_dp.reshape(-1, 1)
).reshape(lstm_pred_168_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


{'MAE': 2093.406195731432,
 'MSE': 8107396.7998581575,
 'RMSE': np.float64(2847.34908289415),
 'MAPE': 6.8988881241233635,
 'R2': 0.8062684460954157,
 'Bias': np.float64(792.6300895236636)}

In [35]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dropout(0.35),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [36]:
lstm_model_dp = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_dp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_dp.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1597 - mae: 0.2987 - val_loss: 0.1023 - val_mae: 0.2452
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0966 - mae: 0.2373 - val_loss: 0.0822 - val_mae: 0.2173
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0869 - mae: 0.2246 - val_loss: 0.0815 - val_mae: 0.2160
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0812 - mae: 0.2169 - val_loss: 0.0761 - val_mae: 0.2068
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0771 - mae: 0.2112 - val_loss: 0.0782 - val_mae: 0.2131
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0738 - mae: 0.2067 - val_loss: 0.0843 - val_mae: 0.2191
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0706 - mae: 0.2021 - val_loss: 0.0807 - val_mae: 0.2120
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0686 - mae: 0.1993 - val_loss: 0.0834 - val_mae: 0.2196
Epoch 9/10
1584/1584 ━━━━━━━━━

In [39]:
lstm_pred_168_dp = lstm_model_dp.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_dp.reshape(-1, 1)
).reshape(lstm_pred_168_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2102.219668501171,
 'MSE': 8191632.822401408,
 'RMSE': np.float64(2862.102867194226),
 'MAPE': 6.837262466515703,
 'R2': 0.8042555711930387,
 'Bias': np.float64(375.5403306296265)}

In [42]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [43]:
lstm_model_dp = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_dp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_dp.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1844 - mae: 0.3252 - val_loss: 0.1013 - val_mae: 0.2444
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1181 - mae: 0.2635 - val_loss: 0.1074 - val_mae: 0.2560
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1059 - mae: 0.2492 - val_loss: 0.0869 - val_mae: 0.2237
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1000 - mae: 0.2414 - val_loss: 0.0833 - val_mae: 0.2188
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0943 - mae: 0.2346 - val_loss: 0.0863 - val_mae: 0.2232
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0901 - mae: 0.2288 - val_loss: 0.0873 - val_mae: 0.2228
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0869 - mae: 0.2246 - val_loss: 0.0846 - val_mae: 0.2171
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0837 - mae: 0.2206 - val_loss: 0.0903 - val_mae: 0.2261
Epoch 9/10
1584/1584 ━━━━━━━━━━

In [44]:
lstm_pred_168_dp = lstm_model_dp.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_dp.reshape(-1, 1)
).reshape(lstm_pred_168_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 1969.0540437155278,
 'MSE': 7186750.671312515,
 'RMSE': np.float64(2680.8115695275033),
 'MAPE': 6.300087888790882,
 'R2': 0.828267888022633,
 'Bias': np.float64(-291.70461006048976)}

# Adding Batch Normaliation

In [45]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [46]:
lstm_model_bn = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_bn.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.2574 - mae: 0.3787 - val_loss: 0.1187 - val_mae: 0.2691
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1368 - mae: 0.2849 - val_loss: 0.0977 - val_mae: 0.2410
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1233 - mae: 0.2697 - val_loss: 0.0880 - val_mae: 0.2247
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1134 - mae: 0.2583 - val_loss: 0.0901 - val_mae: 0.2281
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1083 - mae: 0.2522 - val_loss: 0.0929 - val_mae: 0.2323
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1034 - mae: 0.2462 - val_loss: 0.0900 - val_mae: 0.2278
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0993 - mae: 0.2413 - val_loss: 0.0880 - val_mae: 0.2261
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0961 - mae: 0.2373 - val_loss: 0.0888 - val_mae: 0.2275
Epoch 9/10
1584/1584 ━━━

In [47]:
lstm_pred_168_bn = lstm_model_bn.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_bn.reshape(-1, 1)
).reshape(lstm_pred_168_bn.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2070.2316370186686,
 'MSE': 7643918.283221489,
 'RMSE': np.float64(2764.7636939206013),
 'MAPE': 6.864982824811837,
 'R2': 0.8173435686589227,
 'Bias': np.float64(790.5956146045879)}

# New OPtimizers

In [62]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),


        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [53]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)


In [55]:

history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 1.2169 - mae: 0.8538 - val_loss: 0.4756 - val_mae: 0.5451
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.7415 - mae: 0.6739 - val_loss: 0.3599 - val_mae: 0.4740
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.6045 - mae: 0.6089 - val_loss: 0.3104 - val_mae: 0.4407
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.5264 - mae: 0.5691 - val_loss: 0.2821 - val_mae: 0.4202
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.4748 - mae: 0.5406 - val_loss: 0.2648 - val_mae: 0.4071
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.4379 - mae: 0.5193 - val_loss: 0.2515 - val_mae: 0.3965
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.4094 - mae: 0.5021 - val_loss: 0.2421 - val_mae: 0.3886
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.3877 - mae: 0.4885 - val_loss: 0.2348 - val_mae: 0.3826
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2817.8998274735054,
 'MSE': 13025983.907269912,
 'RMSE': np.float64(3609.1527963318363),
 'MAPE': 9.57642785472432,
 'R2': 0.6887355872928707,
 'Bias': np.float64(604.3252836178002)}

In [56]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.2350 - mae: 0.3642 - val_loss: 0.1197 - val_mae: 0.2652
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1374 - mae: 0.2853 - val_loss: 0.1210 - val_mae: 0.2735
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1240 - mae: 0.2703 - val_loss: 0.0958 - val_mae: 0.2347
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1158 - mae: 0.2611 - val_loss: 0.1040 - val_mae: 0.2512
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1111 - mae: 0.2556 - val_loss: 0.1117 - val_mae: 0.2457
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1058 - mae: 0.2492 - val_loss: 0.1021 - val_mae: 0.2420
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1020 - mae: 0.2446 - val_loss: 0.0958 - val_mae: 0.2394
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0987 - mae: 0.2407 - val_loss: 0.1064 - val_mae: 0.2505
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2025.5845268342443,
 'MSE': 7116907.695226848,
 'RMSE': np.float64(2667.7533047916645),
 'MAPE': 6.561243845127129,
 'R2': 0.8299368316577383,
 'Bias': np.float64(14.304081307266548)}

# Changing Learning Rate 

In [57]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1969 - mae: 0.3351 - val_loss: 0.2340 - val_mae: 0.3696
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1374 - mae: 0.2837 - val_loss: 0.1251 - val_mae: 0.2727
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1269 - mae: 0.2725 - val_loss: 0.1141 - val_mae: 0.2563
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1213 - mae: 0.2664 - val_loss: 0.1265 - val_mae: 0.2760
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1173 - mae: 0.2622 - val_loss: 0.1221 - val_mae: 0.2615
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1153 - mae: 0.2597 - val_loss: 0.1326 - val_mae: 0.2804
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1138 - mae: 0.2578 - val_loss: 0.1197 - val_mae: 0.2664
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1124 - mae: 0.2564 - val_loss: 0.1147 - val_mae: 0.2592
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2286.1492992531557,
 'MSE': 8918676.523530735,
 'RMSE': np.float64(2986.4153300454936),
 'MAPE': 7.491368469178184,
 'R2': 0.7868823860075315,
 'Bias': np.float64(377.8933612000542)}

In [58]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.2965 - mae: 0.4082 - val_loss: 0.1296 - val_mae: 0.2775
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1511 - mae: 0.3003 - val_loss: 0.1080 - val_mae: 0.2524
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1342 - mae: 0.2823 - val_loss: 0.0971 - val_mae: 0.2378
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1251 - mae: 0.2718 - val_loss: 0.1003 - val_mae: 0.2443
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1190 - mae: 0.2649 - val_loss: 0.0913 - val_mae: 0.2313
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1145 - mae: 0.2596 - val_loss: 0.0910 - val_mae: 0.2302
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1106 - mae: 0.2550 - val_loss: 0.1059 - val_mae: 0.2474
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1078 - mae: 0.2517 - val_loss: 0.0935 - val_mae: 0.2324
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2055.2365116455967,
 'MSE': 7291163.783825497,
 'RMSE': np.float64(2700.2155069226415),
 'MAPE': 6.894155457428436,
 'R2': 0.8257728683468342,
 'Bias': np.float64(756.8230088590758)}

# Early Stoping

In [59]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [60]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    callbacks = [early_stopping],
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1944 - mae: 0.3331 - val_loss: 0.1567 - val_mae: 0.3085
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1376 - mae: 0.2838 - val_loss: 0.1289 - val_mae: 0.2804
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1280 - mae: 0.2736 - val_loss: 0.1226 - val_mae: 0.2735
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1226 - mae: 0.2682 - val_loss: 0.1325 - val_mae: 0.2851
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1180 - mae: 0.2630 - val_loss: 0.1703 - val_mae: 0.3264
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1150 - mae: 0.2598 - val_loss: 0.1406 - val_mae: 0.2831
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1115 - mae: 0.2556 - val_loss: 0.2089 - val_mae: 0.3685
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1114 - mae: 0.2552 - val_loss: 0.1482 - val_mae: 0.2920
674/674 ━━━━━━━━━━━━━━━━━━━━ 2s

{'MAE': 2400.126212042043,
 'MSE': 9519815.741643934,
 'RMSE': np.float64(3085.4198647256962),
 'MAPE': 8.027378691509938,
 'R2': 0.7725177708649626,
 'Bias': np.float64(789.793445600131)}

# New Batch Size

In [61]:
history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=32
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.1476 - mae: 0.2947 - val_loss: 0.1442 - val_mae: 0.2992
Epoch 2/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1450 - mae: 0.2921 - val_loss: 0.1394 - val_mae: 0.2919
Epoch 3/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1507 - mae: 0.2974 - val_loss: 0.1387 - val_mae: 0.2871
Epoch 4/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1491 - mae: 0.2961 - val_loss: 0.1424 - val_mae: 0.2931
Epoch 5/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1519 - mae: 0.2983 - val_loss: 0.1313 - val_mae: 0.2829
Epoch 6/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.2114 - mae: 0.3553 - val_loss: 0.2217 - val_mae: 0.3668
Epoch 7/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.3149 - mae: 0.4383 - val_loss: 0.2608 - val_mae: 0.3981
Epoch 8/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.3644 - mae: 0.4734 - val_loss: 0.3150 - val_mae: 0.4431
Epoch 9/10
3167/3167 ━━━━━━━━━━━

{'MAE': 3252.461922991529,
 'MSE': 17352189.20721325,
 'RMSE': np.float64(4165.59590061413),
 'MAPE': 10.626736239941604,
 'R2': 0.5853580795726447,
 'Bias': np.float64(-342.31112114060664)}

In [63]:
history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=16
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5163 - mae: 0.5604 - val_loss: 0.4079 - val_mae: 0.5084
Epoch 2/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5285 - mae: 0.5692 - val_loss: 0.4212 - val_mae: 0.5136
Epoch 3/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5445 - mae: 0.5783 - val_loss: 0.4169 - val_mae: 0.5153
Epoch 4/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.5424 - mae: 0.5788 - val_loss: 0.4409 - val_mae: 0.5316
Epoch 5/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.5343 - mae: 0.5748 - val_loss: 0.4073 - val_mae: 0.5056
Epoch 6/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.5370 - mae: 0.5762 - val_loss: 0.4121 - val_mae: 0.5071
Epoch 7/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5438 - mae: 0.5784 - val_loss: 0.4231 - val_mae: 0.5123
Epoch 8/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5370 - mae: 0.5738 - val_loss: 0.4247 - val_mae: 0.5173
Epoch 9/10
6334/6334 ━━━━━━━━━━━

{'MAE': 3382.526341136323,
 'MSE': 18625296.021118943,
 'RMSE': np.float64(4315.703421357744),
 'MAPE': 11.257215633140486,
 'R2': 0.5549363588362453,
 'Bias': np.float64(155.9917540118888)}

# Adding Additional Layers

In [69]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([
        
        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(
            64,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.LSTM(
            32
        ),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(32, activation="relu"),

        tf.keras.layers.Dense(1)
    ])

    return model

In [70]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.5911 - mae: 0.6157 - val_loss: 0.4709 - val_mae: 0.5407
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5661 - mae: 0.6049 - val_loss: 0.4745 - val_mae: 0.5425
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5587 - mae: 0.6018 - val_loss: 0.4880 - val_mae: 0.5450
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5543 - mae: 0.6001 - val_loss: 0.4847 - val_mae: 0.5406
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5513 - mae: 0.5989 - val_loss: 0.4826 - val_mae: 0.5410
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5489 - mae: 0.5978 - val_loss: 0.5067 - val_mae: 0.5539
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5467 - mae: 0.5968 - val_loss: 0.5121 - val_mae: 0.5573
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5449 - mae: 0.5960 - val_loss: 0.5270 - val_mae: 0.5663
Epoch 9/10
1584/1584 ━━━

ValueError: Found input variables with inconsistent numbers of samples: [517536, 21564]

In [77]:
print("y_test_original shape:", y_test_original.shape)
print("lstm_pred_168_no shape:", lstm_pred_168_no.shape)
print("lstm_pred_original_168 shape:", lstm_pred_original_168.shape)

print("y_test_original size:", np.asarray(y_test_original).size)
print("prediction size:", np.asarray(lstm_pred_original_168).size)

y_test_original shape: (21564, 24, 1)
lstm_pred_168_no shape: (21564, 1)
lstm_pred_original_168 shape: (21564, 1)
y_test_original size: 517536
prediction size: 21564


In [78]:
y_test_original_168 = y_test_original[:, -1, :]

print(y_test_original_168.shape)
print(lstm_pred_original_168.shape)

(21564, 1)
(21564, 1)


In [80]:
lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original_168,
    lstm_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


{'MAE': 4056.809134835548,
 'MSE': 27195216.37617009,
 'RMSE': np.float64(5214.903294996954),
 'MAPE': 13.924634126834937,
 'R2': 0.3507844731481431,
 'Bias': np.float64(1303.0323106146238)}

In [27]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([
        
        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(
            128,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.LSTM(
            64
        ),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(32, activation="relu"),

        tf.keras.layers.Dense(1)
    ])

    return model

In [28]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)


E0000 00:00:1786531845.478729  593804 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1786531850.597790  593804 cpu_allocator_impl.cc:82] Allocation of 1293781440 exceeds 10% of free system memory.


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 0s 403ms/step - loss: 0.5810 - mae: 0.6117

W0000 00:00:1786532495.698178  593804 cpu_allocator_impl.cc:82] Allocation of 275316384 exceeds 10% of free system memory.


1584/1584 ━━━━━━━━━━━━━━━━━━━━ 726s 456ms/step - loss: 0.5810 - mae: 0.6117 - val_loss: 0.4758 - val_mae: 0.5357
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 894s 564ms/step - loss: 0.5607 - mae: 0.6027 - val_loss: 0.4709 - val_mae: 0.5378
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 777s 491ms/step - loss: 0.5550 - mae: 0.6004 - val_loss: 0.4730 - val_mae: 0.5353
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 699s 442ms/step - loss: 0.5511 - mae: 0.5985 - val_loss: 0.4705 - val_mae: 0.5366
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 673s 425ms/step - loss: 0.5479 - mae: 0.5971 - val_loss: 0.4668 - val_mae: 0.5335
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 942s 595ms/step - loss: 0.5450 - mae: 0.5959 - val_loss: 0.4777 - val_mae: 0.5352
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 841s 505ms/step - loss: 0.5427 - mae: 0.5948 - val_loss: 0.4840 - val_mae: 0.5375
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 515s 325ms/step - loss: 0.5403 - mae: 0.5938 - val_loss: 0.4704 - val_mae: 0.5356
Epoch 9/10
1584/158

In [86]:
lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original_168,
    lstm_pred_original_168
)

results




674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


{'MAE': 4099.808879780827,
 'MSE': 27722606.824187826,
 'RMSE': np.float64(5265.226189271248),
 'MAPE': 14.176232914515655,
 'R2': 0.3381943888174852,
 'Bias': np.float64(1595.3570324093396)}

In [27]:
def build_lstm_hyper(hp):

    model = Sequential()

    model.add(
        LSTM(
            units=hp.Choice(
                "lstm_units",
                values=[32, 64, 128]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    # Next 24 hours
    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [28]:
tuner_lstm = kt.RandomSearch(
    build_lstm_hyper,
    objective="val_loss",
    max_trials=5,
    directory="hyperparameter_tuning",
    project_name="bilstm_48_to_24"
)


tuner_lstm.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64,
    verbose=1
)

Trial 5 Complete [00h 02m 37s]
val_loss: 0.07770843803882599

Best val_loss So Far: 0.0770488828420639
Total elapsed time: 00h 12m 32s


In [29]:
best_hp_lstm = tuner_lstm.get_best_hyperparameters(
    num_trials=1
)[0]

best_hp_lstm

In [31]:
best_lstm_model = tuner_lstm.get_best_models(
    num_models=1
)[0]

best_lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        75,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,592 (334.34 KB)

 Trainable params: 85,592 (334.34 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history_best_lstm = best_lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0328 - mae: 0.1346 - val_loss: 0.0990 - val_mae: 0.2390
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0292 - mae: 0.1276 - val_loss: 0.0837 - val_mae: 0.2121
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0262 - mae: 0.1214 - val_loss: 0.0838 - val_mae: 0.2132
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0239 - mae: 0.1162 - val_loss: 0.0881 - val_mae: 0.2176
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0218 - mae: 0.1113 - val_loss: 0.0888 - val_mae: 0.2199
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0202 - mae: 0.1074 - val_loss: 0.0812 - val_mae: 0.2050
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0187 - mae: 0.1035 - val_loss: 0.0861 - val_mae: 0.2143
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0175 - mae: 0.1003 - val_loss: 0.0814 - val_mae: 0.2049
Epoch 9/10
1584/1584 ━━━━━━━

In [50]:
y_test_original_168 = y_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

y_test_original_168 = y_test_original_168.squeeze(-1)

In [51]:
lstm_pred_168_hy = best_lstm_model.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_hy.reshape(-1, 1)
).reshape(lstm_pred_168_hy.shape)


674/674 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [52]:
results = evaluate_deep_model(
    y_test_original_168,
    lstm_pred_original_168
)
results

{'MAE': 1947.5263563790731,
 'MSE': 6851497.791196226,
 'RMSE': np.float64(2617.5365883204436),
 'MAPE': 6.4726604688660565,
 'R2': 0.8362789750607131,
 'Bias': np.float64(696.291991417626)}

# 168 RNN

In [24]:
sequence_length = 168
forecast_horizon = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [25]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [26]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



NameError: name 'build_lstm_model' is not defined

In [ ]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

results


# Drop out

In [ ]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [ ]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



In [ ]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

results


In [ ]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dropout(0.35),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [ ]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



In [ ]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

results


In [ ]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [ ]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)


In [ ]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

results


# Batch Normaliztion

In [ ]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [ ]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



In [ ]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

results


# New Optimizers

In [ ]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



In [ ]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

results
